## Step 4 : Player curves and the draft top 10s

Reads `outputs/balls_clean.parquet`, `outputs/balls_pressure.parquet` and `outputs/balls_value.parquet` (steps 1 to 3).
Writes `outputs/04_batters.csv` and `outputs/04_bowlers.csv` (every eligible player, ranked).

For every player I measure two things:
1. **Level** : value per ball over all his balls (runs above an average player in the same situation, wickets counted as runs)
2. **Clutch** : how much better or worse he is on high pressure balls than on his other balls

Both are shrunk towards zero. The fewer balls a player has, the harder the pull, so one lucky over cannot put anyone at rank 1.
The score is the value I expect from him on a high pressure ball = level + clutch.

In [1]:
# Importing Libraries
import os
import numpy as np
import pandas as pd

output_folder = os.path.join("..", "outputs")
keys = ['match_id', 'innings', 'innings_ball_num']

# Loading Dataset (the 3 tables from steps 1 to 3, one row per ball)
balls = pd.read_parquet(os.path.join(output_folder, "balls_clean.parquet"), columns=keys + [
    'striker_id', 'striker_name', 'bowler_id', 'bowler_name', 'batting_team_id', 'bowling_team_id',
    'legal', 'comp_group', 'pace_or_spin'])
balls = balls.merge(pd.read_parquet(os.path.join(output_folder, "balls_pressure.parquet"), columns=keys + ['pressure']), on=keys)
balls = balls.merge(pd.read_parquet(os.path.join(output_folder, "balls_value.parquet"), columns=keys + ['batter_value', 'bowler_value']), on=keys)

# The 30 Other T20 matches were only used to train the models, their players are not ranked
balls = balls[balls['comp_group'] != 'Other T20']
print(balls.shape)

(193988, 15)


In [2]:
# Does pressure change the value of an average player? (10 groups of pressure, both should stay near 0)
# If it did, every player's clutch number would be shifted by the same amount
balls['pressure_group'] = (balls['pressure'] // 10).clip(upper=9).astype(int)
print(balls.groupby('pressure_group').agg(
    batters=('batter_value', 'mean'), bowlers=('bowler_value', 'mean'), balls=('pressure', 'size')).round(3).T)

# High pressure ball = pressure 80 or more, the top fifth of all balls
balls['high'] = balls['pressure'] >= 80
print("share of high pressure balls:", round(balls['high'].mean(), 3))

pressure_group          0          1          2          3          4  \
batters             0.032      0.017     -0.008     -0.016     -0.015   
bowlers            -0.016     -0.017      0.006      0.015      0.012   
balls           19144.000  19512.000  19515.000  19533.000  19539.000   

pressure_group          5          6          7          8          9  
batters             0.019      0.009     -0.027      0.011     -0.001  
bowlers            -0.021     -0.013      0.023     -0.011     -0.003  
balls           19487.000  19378.000  19391.000  19410.000  19079.000  
share of high pressure balls: 0.198


In [3]:
# Level and clutch of every player, shrunk towards 0
#
# level  = his average value per ball over all balls
# delta  = his average on high pressure balls minus his average on the other balls
# Shrinking : a raw average = true skill + luck. If the true skill varies across players by tau^2 and the luck in
# his average is L, the best guess of his skill is raw * tau^2 / (tau^2 + L).
# tau^2 is measured from the players with enough high pressure balls : spread of the raw numbers minus the average luck.
# If tau^2 comes out as 0, the spread is all luck and every player's number is pulled all the way to 0.
#
# Luck : spread of a single ball / number of balls, times one factor for the fact that balls in the same match
# share the pitch and conditions and are not independent. The factor = luck measured match by match / luck measured
# ball by ball, averaged over the players in the pool (about 1.1). One factor for everyone, because a player's own
# match by match luck from 10 to 30 matches is too noisy and reshuffled the list by itself.
#
# Value on a high pressure ball = level + share of his balls that are NOT high pressure * delta
# (because his overall average already has the high pressure balls in it)
def player_curves(balls, player, name, team, value, min_high):
    per_player = balls.groupby(player)
    table = pd.DataFrame({
        'name': per_player[name].first(),
        'team': per_player[team].agg(lambda x: x.mode()[0]),
        'balls': per_player.size(),
        'high_balls': balls[balls['high']].groupby(player).size(),
        'matches': per_player['match_id'].nunique(),
        'first_innings_share': per_player['innings'].agg(lambda x: (x == 1).mean()),
        'raw_level': per_player[value].mean(),
        'raw_high': balls[balls['high']].groupby(player)[value].mean(),
        'raw_other': balls[~balls['high']].groupby(player)[value].mean(),
    }).fillna({'high_balls': 0})
    table['delta'] = table['raw_high'] - table['raw_other']
    other_balls = table['balls'] - table['high_balls']
    in_pool = table['high_balls'] >= min_high

    # Ball by ball luck
    high = balls['high'].values
    gap_level = balls[value] - per_player[value].transform('mean')
    gap_split = balls[value] - balls.groupby([player, 'high'])[value].transform('mean')
    luck_level = gap_level.var() / table['balls']
    luck_delta = gap_split[high].var() / table['high_balls'] + gap_split[~high].var() / other_balls

    # Match by match luck = sum over his matches of (the match's total gap from his average)^2 / n^2
    def match_luck(rows, gap, n):
        per_match = gap[rows].groupby([balls.loc[rows, player], balls.loc[rows, 'match_id']]).sum()
        return (per_match ** 2).groupby(level=0).sum() / n ** 2

    factor_level = match_luck(np.ones(len(balls), dtype=bool), gap_level, table['balls'])[in_pool].mean() / luck_level[in_pool].mean()
    factor_delta = (match_luck(high, gap_split, table['high_balls']) + match_luck(~high, gap_split, other_balls))[in_pool].mean() \
        / luck_delta[in_pool].mean()
    table['luck_level'] = luck_level * factor_level
    table['luck_delta'] = luck_delta * factor_delta

    pool = table[in_pool]
    tau_level = max(pool['raw_level'].var() - pool['luck_level'].mean(), 0)
    tau_delta = max(pool['delta'].var() - pool['luck_delta'].mean(), 0)

    table['level'] = table['raw_level'] * tau_level / (tau_level + table['luck_level'])
    table['clutch'] = (table['delta'] * tau_delta / (tau_delta + table['luck_delta'])).fillna(0) \
        * (1 - table['high_balls'] / table['balls'])
    table['score'] = table['level'] + table['clutch']

    # Consistency : in the matches where he had 6 or more high pressure balls, how often was he above the average player?
    per_match = balls[balls['high']].groupby([player, 'match_id'])[value].agg(['sum', 'size'])
    per_match = per_match[per_match['size'] >= 6]
    table['pressure_matches'] = per_match.groupby(player).size()
    table['beat_average'] = (per_match['sum'] > 0).groupby(player).mean()
    table = table.fillna({'pressure_matches': 0})

    print(f"{value}: true spread of level {np.sqrt(tau_level):.3f}, true spread of clutch {np.sqrt(tau_delta):.3f}, "
          f"raw spread of clutch {pool['delta'].std():.3f}, luck in it {np.sqrt(pool['luck_delta'].mean()):.3f}, "
          f"match factor {factor_level:.2f} / {factor_delta:.2f} (runs per ball, {len(pool)} players with {min_high}+ high pressure balls)")
    return table

In [4]:
# Batters on legal balls (a wide is not a ball faced), bowlers on every delivery
batters = player_curves(balls[balls['legal']], 'striker_id', 'striker_name', 'batting_team_id', 'batter_value', min_high=100)
bowlers = player_curves(balls, 'bowler_id', 'bowler_name', 'bowling_team_id', 'bowler_value', min_high=120)

# Does the result depend on where high pressure starts? Same numbers with the cut at 70 and at 90
for cut in [70, 90]:
    test = balls.assign(high=balls['pressure'] >= cut)
    print(f"cut at {cut}:")
    player_curves(test[test['legal']], 'striker_id', 'striker_name', 'batting_team_id', 'batter_value', min_high=100)
    player_curves(test, 'bowler_id', 'bowler_name', 'bowling_team_id', 'bowler_value', min_high=120)

batter_value: true spread of level 0.103, true spread of clutch 0.000, raw spread of clutch 0.216, luck in it 0.231, match factor 1.12 / 1.04 (runs per ball, 121 players with 100+ high pressure balls)
bowler_value: true spread of level 0.072, true spread of clutch 0.046, raw spread of clutch 0.211, luck in it 0.206, match factor 1.10 / 1.05 (runs per ball, 102 players with 120+ high pressure balls)
cut at 70:
batter_value: true spread of level 0.109, true spread of clutch 0.000, raw spread of clutch 0.230, luck in it 0.244, match factor 1.20 / 1.11 (runs per ball, 162 players with 100+ high pressure balls)
bowler_value: true spread of level 0.079, true spread of clutch 0.000, raw spread of clutch 0.221, luck in it 0.221, match factor 1.14 / 1.09 (runs per ball, 153 players with 120+ high pressure balls)
cut at 90:
batter_value: true spread of level 0.076, true spread of clutch 0.000, raw spread of clutch 0.195, luck in it 0.212, match factor 1.00 / 0.91 (runs per ball, 47 players with 

In [5]:
# Eligibility : how many players are left for a few minimums of high pressure balls and pressure matches
for label, table, minimums in [('batters', batters, [60, 100, 150]), ('bowlers', bowlers, [60, 120, 180])]:
    for min_high in minimums:
        counts = [int(((table['high_balls'] >= min_high) & (table['pressure_matches'] >= m)).sum()) for m in [5, 10, 15]]
        print(f"{label}: {min_high}+ high pressure balls and 5 / 10 / 15+ pressure matches -> {counts} players")

batters: 60+ high pressure balls and 5 / 10 / 15+ pressure matches -> [150, 67, 30] players
batters: 100+ high pressure balls and 5 / 10 / 15+ pressure matches -> [120, 67, 30] players
batters: 150+ high pressure balls and 5 / 10 / 15+ pressure matches -> [79, 64, 30] players
bowlers: 60+ high pressure balls and 5 / 10 / 15+ pressure matches -> [153, 85, 53] players
bowlers: 120+ high pressure balls and 5 / 10 / 15+ pressure matches -> [102, 85, 53] players
bowlers: 180+ high pressure balls and 5 / 10 / 15+ pressure matches -> [69, 69, 53] players


In [6]:
# My pick : 100+ high pressure balls (batters) or 120+ (bowlers, they bowl more balls in a match),
# and 10+ pressure matches (a match with 6 or more high pressure balls), about one full season of tight finishes.
# Raising the ball minimum barely changes the list, the match minimum is the one that bites.
#
# IPL rule : the report is for the Rajasthan Royals and the IPL 2027 auction, so a player must also have a real
# IPL sample : half of the ball minimum and 5+ pressure matches in the IPL itself. His T20I balls still count as
# evidence, but a player with no IPL record (for example the Pakistan players) cannot be ranked.
team_names = pd.read_csv(os.path.join("..", "teams.csv")).drop_duplicates('team_id').set_index('team_id')['team_name']
columns = ['name', 'team', 'balls', 'high_balls', 'pressure_matches', 'ipl_high_balls', 'ipl_pressure_matches',
           'first_innings_share', 'raw_level', 'raw_high', 'level', 'clutch', 'score', 'beat_average']

ranked = {}
for label, table, min_high, source, player in [
    ('batters', batters, 100, balls[balls['legal']], 'striker_id'),
    ('bowlers', bowlers, 120, balls, 'bowler_id'),
]:
    ipl_high = source[(source['comp_group'] == 'IPL') & source['high']].groupby([player, 'match_id']).size()
    table['ipl_high_balls'] = ipl_high.groupby(level=0).sum()
    table['ipl_pressure_matches'] = (ipl_high >= 6).groupby(level=0).sum()
    table[['ipl_high_balls', 'ipl_pressure_matches']] = table[['ipl_high_balls', 'ipl_pressure_matches']].fillna(0)

    overall = (table['high_balls'] >= min_high) & (table['pressure_matches'] >= 10)
    in_ipl = (table['ipl_high_balls'] >= min_high / 2) & (table['ipl_pressure_matches'] >= 5)
    eligible = table[overall & in_ipl].copy()
    eligible['team'] = eligible['team'].map(team_names)
    eligible = eligible.sort_values('score', ascending=False)
    eligible.insert(0, 'rank', np.arange(1, len(eligible) + 1))
    eligible[['rank'] + columns].round(4).to_csv(os.path.join(output_folder, f"04_{label}.csv"))
    ranked[label] = eligible
    left_out = table[overall & ~in_ipl].sort_values('score', ascending=False)
    print(f"{label}: {len(eligible)} eligible, {len(left_out)} left out by the IPL rule. Best of those: {left_out['name'].head(5).tolist()}")

pd.set_option('display.width', 250)
print(ranked['batters'][['rank'] + columns].head(10).round(3).to_string(index=False))

batters: 57 eligible, 10 left out by the IPL rule. Best of those: ['Reeza Hendricks', 'Glenn Phillips', 'Mohammad Rizwan', 'Daryl Mitchell', 'Sikandar Raza']
bowlers: 62 eligible, 23 left out by the IPL rule. Best of those: ['Mohammad Nawaz', 'Blessing Muzarabani', 'Haris Rauf', 'Taskin Ahmed', 'Abrar Ahmed']
 rank             name                        team  balls  high_balls  pressure_matches  ipl_high_balls  ipl_pressure_matches  first_innings_share  raw_level  raw_high  level  clutch  score  beat_average
    1  Abhishek Sharma         Sunrisers Hyderabad   2239       429.0              28.0           284.0                  16.0                0.546      0.392     0.396  0.293     0.0  0.293         0.643
    2 Suryakumar Yadav                       India   2746       493.0              30.0           273.0                  15.0                0.507      0.301     0.278  0.236    -0.0  0.236         0.700
    3        Tim David                   Australia   1210       293.0        

In [7]:
print(ranked['bowlers'][['rank'] + columns].head(10).round(3).to_string(index=False))

 rank                name                        team  balls  high_balls  pressure_matches  ipl_high_balls  ipl_pressure_matches  first_innings_share  raw_level  raw_high  level  clutch  score  beat_average
    1      Josh Hazlewood Royal Challengers Bengaluru   1515       324.0              22.0           227.0                  17.0                0.487      0.229     0.521  0.117   0.020  0.137         0.773
    2      Jasprit Bumrah              Mumbai Indians   1780       374.0              25.0           304.0                  22.0                0.541      0.262     0.018  0.144  -0.020  0.125         0.520
    3 Matheesha Pathirana         Chennai Super Kings   1279       275.0              17.0           190.0                  12.0                0.457      0.235     0.300  0.110   0.004  0.114         0.824
    4        Sunil Narine       Kolkata Knight Riders   1553       311.0              23.0           311.0                  23.0                0.519      0.181     0.374  

In [8]:
# Checks on the draft lists
# 1. How far does shrinking move a player? Raw high pressure value against the score, rank correlation
# 2. Is any one team taking many places? (a player in a strong team could start with a small head start)
# 3. Batters who bat first get fewer high pressure balls, is the top 10 tilted to chasers?
for label in ['batters', 'bowlers']:
    table = ranked[label]
    top = table.head(10)
    print(f"{label}: rank correlation of score with raw high pressure value {table['score'].corr(table['raw_high'], method='spearman'):.2f}, "
          f"with raw level {table['score'].corr(table['raw_level'], method='spearman'):.2f}")
    print(f"   teams in the top 10: {top['team'].value_counts().to_dict()}")
    print(f"   share of balls in the first innings: top 10 {top['first_innings_share'].mean():.2f}, all eligible {table['first_innings_share'].mean():.2f}")

batters: rank correlation of score with raw high pressure value 0.70, with raw level 0.99
   teams in the top 10: {'Sunrisers Hyderabad': 3, 'India': 1, 'Australia': 1, 'England': 1, 'Royal Challengers Bengaluru': 1, 'Gujarat Titans': 1, 'Rajasthan Royals': 1, 'Lucknow Super Giants': 1}
   share of balls in the first innings: top 10 0.55, all eligible 0.52
bowlers: rank correlation of score with raw high pressure value 0.83, with raw level 0.97
   teams in the top 10: {'Kolkata Knight Riders': 2, 'Royal Challengers Bengaluru': 1, 'Mumbai Indians': 1, 'Chennai Super Kings': 1, 'Australia': 1, 'Punjab Kings': 1, 'Sunrisers Hyderabad': 1, 'New Zealand': 1, 'Bangladesh': 1}
   share of balls in the first innings: top 10 0.53, all eligible 0.51


### What step 4 found

- **An average player does not change under pressure.** Value stays within 0.03 runs per ball of 0 in every pressure group, so the step 3 baseline is fair at all pressure levels.
- **Batters : clutch is not a skill I can see in this data.** The spread of batters' high pressure minus other balls (0.216 per ball) is no bigger than the luck in it (0.231). The best guess of how a batter does under pressure is his overall level, so the batter list is ranked on level. Same answer with the cut at 70 or 90.
- **Bowlers : clutch is weak and fragile.** With the cut at 80 the true spread is 0.046 per ball (level 0.072), but at 70 and at 90 it is 0. So it only moves bowlers a little (Hazlewood up, Bumrah down) and I do not lean on it. Step 5's split-half test decides if even that stays.
- **IPL rule.** The report is for the Rajasthan Royals and the IPL 2027 auction, so only players with a real IPL sample are ranked. It leaves out 10 batters and 23 bowlers, and changes one name in each top 10 (Mitchell Marsh in for Reeza Hendricks, Mustafizur Rahman in for Mohammad Nawaz).
- Draft top 10s are in the cells above and in `outputs/04_batters.csv` and `outputs/04_bowlers.csv`. Step 5 tests if they hold.
- Surprising name : Rajat Patidar is 6th on level (0.295) but only 0.069 on his 132 high pressure balls and above average in 3 of his 10 pressure matches. With clutch at 0 for batters, those 132 balls are treated as luck. Step 5's split-half test decides if that is right.